#  Entorno de Evaluación  


## Dependencias y Funciones

In [ ]:
import requests
import time
from dotenv import load_dotenv
from IPython.display import display
import pandas as pd

load_dotenv()
print(" Imports OK")


API_URL   = "http://localhost:8000"   
TIMEOUT   = 10000                       
PAUSA_SEG = 30                         


def ping_api():
    """Verifica que la API esté activa antes de lanzar la evaluación."""
    try:
        r = requests.get(f"{API_URL}/docs", timeout=5)
        print(f" API activa en {API_URL}  (status {r.status_code})")
    except Exception as e:
        print(f" No se pudo conectar con la API: {e}")

ping_api()

In [ ]:
def evaluar_preguntas(df_eval, API_URL, TIMEOUT, PAUSA_SEG):

    outputs = []
    results = []

    print(f"Enviando {len(df_eval)} preguntas a la API...\n")

    for i, row in df_eval.iterrows():
        pregunta = row["input"]
        print(f"  [{i+1}/{len(df_eval)}] {pregunta[:70]}")

        # Pregunta
        try:
            resp = requests.post(
                f"{API_URL}/preguntar",
                json={"pregunta": pregunta},
                timeout=TIMEOUT,
            )
            if resp.status_code == 504:
                print("       Timeout del agente (servidor canceló la tarea)")
                outputs.append("Timeout")
                results.append("Timeout")
                continue
            resp.raise_for_status()
            output = resp.json().get("respuesta", "")
            print(f"       {len(output)} chars")

        except requests.exceptions.Timeout:
            print("       Timeout en pregunta, saltando...")
            outputs.append("Timeout")
            results.append("Timeout")
            continue
        except Exception as e:
            print(f"       Error en pregunta: {e}")
            outputs.append(f"Error: {e}")
            results.append(f"Error: {e}")
            continue

        outputs.append(output)

        # Evaluación
        try:
            eval_resp = requests.post(
                f"{API_URL}/evaluar",
                json={"pregunta": pregunta, "respuesta": str(output)},
                timeout=TIMEOUT,
            )
            eval_resp.raise_for_status()
            veredicto = eval_resp.json().get("veredicto", "Sin veredicto")
        except requests.exceptions.Timeout:
            print("       Timeout en evaluación, saltando...")
            veredicto = "Timeout"
        except Exception as e:
            print(f"       Error evaluando: {e}")
            veredicto = f"Error: {e}"

        results.append(veredicto)
        print(f"       Veredicto: {veredicto[:60]}")

        if i < len(df_eval) - 1:
            time.sleep(PAUSA_SEG)

    df_eval["output"] = outputs
    df_eval["evaluacion_llm"] = results

    print(f"\n{len(outputs)}/{len(df_eval)} preguntas procesadas y evaluadas")
    display(df_eval[["input", "output", "evaluacion_llm"]])

    return df_eval

In [ ]:
def guardar_resultados(df_eval, output_path="resultados_evaluacion.json"):
    df_eval[["input", "output", "evaluacion_llm"]].to_json(
        output_path,
        orient="records",
        force_ascii=False,
        indent=4,
    )
    print(f"\nResultados guardados en '{output_path}'")

## Preguntas básicas

In [ ]:
data = [
    {
    "input": "Obten los nombres de los empleados de northwind",
    },
    {
    "input": "Obten los titulos de los shows con fecha de salida antes del 2010 de la base de datos netflix",
    },
    {
    "input": "Obten el nombre de los alumnos de la base de datos escolar",
    },
    {
    "input": "Dame los nombres del staff de la base de datos dvd",
    },    
]

df_eval = pd.DataFrame(data)

In [ ]:
evaluar_preguntas(df_eval, API_URL, TIMEOUT, PAUSA_SEG)

In [ ]:
guardar_resultados(df_eval,output_path="resultados_basicos_evaluacion.json")

## preguntas avanzadas

In [ ]:
data = [
    {
    "input": "Obten los emails de los clientes que mas gastaron en la base de datos dvd",
    },
    {
    "input": "Obten el titulo de los shows de netflix por la fecha en la que se ha añadido de mas reciente a mas antiguo",
    },
    {
    "input": "Obten las asignaturas que imparte cada profesor y su número de créditos",
    },
    {
    "input": "Dame el el nombre de los proveedores con los productos más vendidos",
    },    
]

df_eval = pd.DataFrame(data)

In [ ]:
evaluar_preguntas(df_eval, API_URL, TIMEOUT, PAUSA_SEG)

In [ ]:
guardar_resultados(df_eval,output_path="resultados_avanzados_evaluacion.json")

## Preguntas difíciles

In [ ]:
data = [
    {
    "input": "Obten el número de alquileres realizados de cada película el mes de mayo",
    },
    {
    "input": "Obten los nombres de los shows junto a su duración y su director ",
    },
    {
    "input": "Obten los nombres de los alumnos asignados a las asignaturas y el nombre de los profesores que las imparten junto a su número de créditos",
    },
    {
    "input": "Dame el el nombre de los proveedores con los productos más vendidos de menor a mayor el mes de septiembre",
    },    
]

df_eval = pd.DataFrame(data)

In [ ]:
evaluar_preguntas(df_eval, API_URL, TIMEOUT, PAUSA_SEG)

In [ ]:
guardar_resultados(df_eval,output_path="resultados_dificiles_evaluacion.json")